# Choropleths and Classification

The [previous section](map_3.ipynb) coloured features by a **category**, where the colours have no order and the only question is which hue goes with which class. This one colours them by a **number**, and that raises a question categories never had to answer: **where do the boundaries between the classes go?**

A map that shades areas by a value is a **choropleth**, and we have built them already – the café density grid in the [third module](../module_3/geoprocessing_4.ipynb), the year of construction by district. What we have never done is choose the class breaks deliberately. GeoPandas has been picking them, and the choice changes the map more than almost anything else you can do to it.

> The other rule of choropleths – that they take **relative** values, never absolute counts – is set out in the [third module](../module_3/geoprocessing_4.ipynb) and not repeated here. It is why this section maps population **density** rather than population.


## 0. Importing Libraries


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import mapclassify as mc
import numpy as np
import rasterio
from rasterstats import zonal_stats

- [**mapclassify**](https://pysal.org/mapclassify/) (`mapclassify`) – the library that computes class breaks. GeoPandas already uses it under the hood whenever you pass `scheme=`; here we also call it directly, to see what it produces.


## 1. Preparing the Data

We need a number per district, and it has to be a relative one. So we take the population from the **WorldPop raster** of the [fifth module](../module_5/rasters_2.ipynb) and divide it by area – exactly the zonal-statistics operation from that section, applied to districts instead of a grid.

_Every dataset and its source is listed on the [Course Modules](../module_0/syllabus.md) page._


In [ ]:
districts = gpd.read_file("../../data/vienna/vienna_admin.gpkg", layer="district")

raster_path = "../../data/austria/vienna_cropped_population_utm.tif"

# the zones must be in the raster's CRS, which is also metric - so area works too
with rasterio.open(raster_path) as src:
    districts = districts.to_crs(src.crs)

stats = zonal_stats(districts, raster_path, stats=["sum"])
districts["population"] = [s["sum"] for s in stats]

districts["area_km2"] = districts.geometry.area / 1_000_000
districts["density"] = districts["population"] / districts["area_km2"]

print(f"{districts['population'].sum():,.0f} people across {len(districts)} districts")
districts[["NAME", "population", "area_km2", "density"]].round(0).head()

### 1.1. Look at the Distribution First

Choosing a classification method before looking at the data is guesswork. The histogram shows the shape, the box plot shows the median and the outliers, and between them they tell you which methods stand a chance.


In [ ]:
fig, (ax_hist, ax_box) = plt.subplots(1, 2, figsize=(12, 4))

districts["density"].plot.hist(bins=15, color="#8856a7", edgecolor="white", ax=ax_hist)
ax_hist.set_title("Distribution of population density")
ax_hist.set_xlabel("People per km²")
ax_hist.set_ylabel("Districts")
ax_hist.spines[["top", "right"]].set_visible(False)

districts["density"].plot.box(ax=ax_box, color="#8856a7")
ax_box.set_title("The same, as a box plot")
ax_box.set_ylabel("People per km²")
ax_box.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

districts["density"].describe().round(0)

Two things to carry forward.

The range is enormous: **Hietzing** holds about 1,600 people per km², **Margareten** about 26,400 – a factor of sixteen between the emptiest and the densest district of the same city.

And the distribution is **skewed to the right**: most districts sit at the low end, with a thin tail of very dense inner ones. That skew is what will separate the methods below. On a symmetric distribution most of them agree; on this one they do not.


## 2. A Map With No Classes at All

Given a numeric column and no `scheme`, GeoPandas maps the values onto a **continuous** colour ramp: the colour bar runs smoothly from the minimum to the maximum, and every value gets its own shade.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

districts.plot(column="density", cmap="BuPu", linewidth=0.8,
               edgecolor="0.75", legend=True, ax=ax,
               legend_kwds={"label": "People per km²"})

ax.set_title("Population density (continuous scale)")
ax.axis("off")
plt.tight_layout()
plt.show()

This is honest – nothing has been grouped – but it is hard to read. The eye cannot put a number to a shade, and because the scale is stretched by the densest districts, the fifteen at the low end are all much the same pale colour.

Classification trades that smooth scale for a handful of classes you can name in a legend. What it costs is the decision we now have to make.


## 3. How Classification Works

### 3.1. The Classifier

A `mapclassify` classifier takes the column and returns two things:

- `bins` – the upper boundary of each class;
- `yb` – the class each feature landed in, numbered from 0.


In [ ]:
classifier = mc.EqualInterval(districts["density"], k=5)

print("Class boundaries:", classifier.bins.round(0))
print("Class of each district:", classifier.yb)
print("Districts per class:", np.bincount(classifier.yb))

### 3.2. The Boundaries on the Histogram

Numbers in a list are hard to judge. Drawn onto the distribution, the same boundaries show immediately whether the classes fit the data or cut across it.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(districts["density"], bins=20, color="#8856a7", edgecolor="white", alpha=0.85)

# every boundary except the last, which is just the maximum
for bound in classifier.bins[:-1]:
    ax.axvline(bound, color="black", linewidth=0.9, linestyle="--", alpha=0.6)

ax.set_xlabel("People per km²")
ax.set_ylabel("Districts")
ax.set_title("Equal Interval: where the class boundaries fall")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### 3.3. The Map

On a map, the method goes into `scheme=`. GeoPandas builds the classifier itself and colours by the class rather than by the raw value – note how the legend changes from a continuous bar to a list of ranges.

Two details in `legend_kwds` are doing work beyond tidiness. `fmt` rounds the boundaries to whole numbers – without it the legend reads `1644.82, 6593.01`, which implies a precision the estimate does not have. And `title` carries the **unit**: a legend of bare numbers leaves the reader guessing whether they are looking at people, people per hectare or people per square kilometre. The unit belongs there once, above the classes, rather than repeated on every row.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

districts.plot(column="density", scheme="EqualInterval", k=5,
               cmap="BuPu", linewidth=0.8, edgecolor="0.75", legend=True, ax=ax,
               legend_kwds={"title": "People per km²", "fmt": "{:.0f}"})

ax.set_title("Equal Interval")
ax.axis("off")
ax.get_legend().set_bbox_to_anchor((1.45, 1))
plt.tight_layout()
plt.show()

### 3.4. One Function for All of Them

Those three steps – classify, draw the boundaries on the histogram, draw the map – are the same for every method. Rather than write them out six times, we wrap them once.

`getattr(mc, scheme)` looks the classifier up by name, so the function takes the method as a string and needs no branch per method. Two of them do need special handling: `StdMean` decides its own number of classes and takes no `k`, and `UserDefined` takes the boundaries instead of computing them.


In [ ]:
def compare_classification(gdf, column, scheme, k=5, bins=None,
                           title=None, unit="People per km²"):
    """Draw the distribution and the map for one classification method."""

    # the classifier, looked up by name
    if scheme == "UserDefined":
        classifier = mc.UserDefined(gdf[column], bins)
    elif scheme == "StdMean":
        classifier = mc.StdMean(gdf[column])      # decides k for itself
    else:
        classifier = getattr(mc, scheme)(gdf[column], k=k)

    fig, (ax_hist, ax_map) = plt.subplots(1, 2, figsize=(14, 5))

    # the histogram, with each bar painted in the colour of its class
    counts, edges, bars = ax_hist.hist(gdf[column], bins=20,
                                       edgecolor="white", alpha=0.9)
    n_classes = len(classifier.bins)
    for bar, left_edge in zip(bars, edges[:-1]):
        which = min(np.searchsorted(classifier.bins, left_edge), n_classes - 1)
        bar.set_facecolor(plt.cm.BuPu((which + 1.5) / (n_classes + 1)))

    for bound in classifier.bins[:-1]:
        ax_hist.axvline(bound, color="black", linewidth=0.9,
                        linestyle="--", alpha=0.6)

    ax_hist.set_xlabel(unit)
    ax_hist.set_ylabel("Districts")
    ax_hist.set_title("Distribution and class boundaries")
    ax_hist.spines[["top", "right"]].set_visible(False)

    # the map, drawn with the same method
    style = dict(column=column, cmap="BuPu", linewidth=0.8,
                 edgecolor="0.75", legend=True, ax=ax_map,
                 # the unit belongs in the legend title, not on every row
                 legend_kwds={"title": unit, "fmt": "{:.0f}"})
    if scheme == "UserDefined":
        gdf.plot(**style, scheme="UserDefined",
                 classification_kwds={"bins": bins})
    elif scheme == "StdMean":
        gdf.plot(**style, scheme="StdMean")
    else:
        gdf.plot(**style, scheme=scheme, k=k)

    ax_map.set_title(title or scheme)
    ax_map.axis("off")
    ax_map.get_legend().set_bbox_to_anchor((1.5, 1))

    plt.tight_layout()
    plt.show()

    print(f"Boundaries: {classifier.bins.round(0)}")
    print(f"Districts per class: {np.bincount(classifier.yb)}")

## 4. The Methods

The six below are the ones in general cartographic use; Slocum et al. (2022) set them out at greater length, with the cases each one suits.

### 4.1. Equal Interval

The range from minimum to maximum is cut into `k` equal slices.

It is the easiest method to explain in a legend – the classes are round, even steps – and it works when the values are spread evenly. Ours are not, and the result shows what that costs.


In [ ]:
compare_classification(districts, "density", "EqualInterval", k=5,
                       title="Equal Interval")

**Ten of the twenty-three districts fall in the first class.** The four densest stretch the range, the even slices follow the range rather than the data, and nearly half the city ends up one shade. The map is not wrong, but it is telling you far less than it could.


### 4.2. Quantiles

The boundaries are placed so that each class holds **the same number of features**.

Every colour is used equally often, so the map looks full and no class is empty. It is also immune to the skew that just defeated Equal Interval.

The price is that the boundaries answer to the ranking rather than to the values: two districts a hair apart can land either side of a line, and two far apart can share a class.


In [ ]:
compare_classification(districts, "density", "Quantiles", k=5,
                       title="Quantiles")

Four or five districts per class, by construction. Look at where the boundaries sit on the histogram, though: several cut straight through the tall bars, splitting districts whose densities are nearly identical.


### 4.3. Natural Breaks (Jenks)

The algorithm looks for the **gaps already in the data**: it minimises the spread inside each class and maximises the difference between them (Jenks, 1967).

That makes the classes describe real structure rather than arithmetic, which is why it is the sensible default for socio-economic values. Two caveats: the boundaries come out as awkward numbers, and they depend on the data, so two maps classified this way **cannot be compared** – each gets its own breaks.


In [ ]:
compare_classification(districts, "density", "NaturalBreaks", k=5,
                       title="Natural Breaks (Jenks)")

The boundaries now sit in the thin parts of the histogram rather than through the peaks. That is the whole idea: cut where the data is already divided.


### 4.4. Standard Deviation

Classes are built outwards from the mean in steps of one standard deviation, so the map shows **departure from the average** rather than the values themselves.

That is a genuinely different question, and a useful one – but it assumes the values are spread symmetrically around the mean. Ours are not, and the method fails in a way worth seeing.


In [ ]:
compare_classification(districts, "density", "StdMean",
                       title="Standard Deviation")

Two things went wrong, and both come from the same assumption.

**The lowest class is empty.** Its boundary sits at roughly **−4,100 people per km²** – a density that cannot exist. The method placed it there because it measures outwards from the mean in both directions, whether or not the data goes that way.

**Fifteen of the twenty-three districts land in the middle class.** Everything within one standard deviation of the mean is grouped together, and on a skewed distribution that is most of the city.

The lesson is not that the method is bad. It is that this method asks a question – _how far from normal is this?_ – which only makes sense when the data has a normal to be far from.


### 4.5. Maximum Breaks

The method sorts the values and puts the boundaries at the `k − 1` **largest gaps** between neighbours.

Where the data falls into clear clusters, this finds them. Where it does not, a single outlier can swallow a whole class.


In [ ]:
compare_classification(districts, "density", "MaximumBreaks", k=5,
                       title="Maximum Breaks")

Twelve districts in the first class, and two classes holding **one district each** – the two densest, each separated from its neighbour by a gap large enough to count. The method did exactly what it promises; this data simply has its biggest gaps at the top.


### 4.6. User Defined

You set the boundaries yourself. Not a statistical method at all, but the right answer in two common situations:

1. **Round numbers.** Take Natural Breaks as a starting point and tidy the boundaries into figures a reader can hold: 5,000 rather than 5,053.
2. **Comparable maps.** Two maps of different years, or of different cities, must use the **same boundaries** – otherwise the colours mean different things on each and the comparison is meaningless. No data-driven method can guarantee that; fixing the bins by hand can.


In [ ]:
# start from what Natural Breaks suggested
natural = mc.NaturalBreaks(districts["density"], k=5)
print("Natural Breaks:", natural.bins.round(0))

# and round it into something a legend can carry
custom_bins = [5000, 10000, 15000, 20000, 27000]

compare_classification(districts, "density", "UserDefined",
                       bins=custom_bins, title="User Defined")

## 5. All of Them at Once

Each method above was a separate map. Set side by side, the disagreement is easier to see: the same district changes class depending only on how the boundaries were drawn.

Below, each row is a method and each column a district, sorted from the emptiest to the densest. Reading down a column shows how differently the same place can be classified.


In [ ]:
methods = {
    "Equal Interval": mc.EqualInterval(districts["density"], k=5),
    "Quantiles": mc.Quantiles(districts["density"], k=5),
    "Natural Breaks": mc.NaturalBreaks(districts["density"], k=5),
    "Maximum Breaks": mc.MaximumBreaks(districts["density"], k=5),
    "Standard Deviation": mc.StdMean(districts["density"]),
}

for name, classifier in methods.items():
    districts[name] = classifier.yb

matrix = (districts.set_index("NAME")
                   .sort_values("density")[list(methods)]
                   .T)

fig, ax = plt.subplots(figsize=(14, 3.5))
ax.imshow(matrix.values, cmap="BuPu", aspect="auto")

ax.set_xticks(range(len(matrix.columns)))
ax.set_xticklabels(matrix.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(matrix.index)))
ax.set_yticklabels(matrix.index)
ax.set_xlabel("Districts, sorted by density →")
ax.set_title("Which class each district lands in, by method")

plt.tight_layout()
plt.show()

Read left to right, every row rises – all five methods agree that Hietzing is emptier than Margareten, as they must. What differs is **where each one decides to change colour**, and that is the whole argument of this section.


## Summary

In this section we built choropleths and chose their classes deliberately.

We learned:

- how to get a mappable **relative** value – population density – by running the zonal statistics of the [fifth module](../module_5/rasters_2.ipynb) over districts;
- why the distribution has to be looked at **before** a method is chosen;
- how a `mapclassify` classifier produces `bins` and `yb`, and how `scheme=` uses them on a map;
- what each of the standard methods does, and the case each one fails on.

If you take one thing away, take this: **the classification method is an analytical decision, not a display setting.** The same twenty-three numbers gave us a map where half the city is one colour, a map where every class is equally full, and a map with an empty class whose boundary lies below zero. Nothing in the code warns you which of those a reader will believe.

As a default, **Natural Breaks** is a reasonable starting point for values like these, and **User Defined** is what you reach for the moment two maps have to be compared.

With that, the map is finished. The [next section](map_2.ipynb) puts it on the web.


## References

Jenks, G. F. (1967). The data model concept in statistical mapping. *International Yearbook of Cartography*, 7, 186–190.

Slocum, T. A., McMaster, R. B., Kessler, F. C., & Howard, H. H. (2022). *Thematic Cartography and Geovisualization* (4th ed.). CRC Press. https://doi.org/10.1201/9781003150527
